# 03. Dummy Classifier — 다음 주 이탈 기준선

`DummyClassifier(strategy="prior")`는 Feature와 Target 사이의 관계를 학습하지 않고 **학습 Fold의 양성률**을 모든 검증 행에 반환합니다. 복잡한 모델은 최소한 이 기준선보다 높은 PR-AUC와 Recall@Top-20%를 보여야 하며, Brier Score와 ECE로 확률 품질도 함께 비교해야 합니다.

학생 한 명이 여러 과목·운영회차·주차 행을 가질 수 있으므로 일반 KFold는 같은 학생을 학습과 검증에 섞을 수 있습니다. 이 노트북은 `id_student` 기준 3-Fold GroupKFold를 사용하며, 각 행이 정확히 한 번 검증된 OOF(out-of-fold) 확률을 결합합니다.

In [ ]:
from importlib.util import module_from_spec, spec_from_file_location
from pathlib import Path
import sys
import pandas as pd

repo_root = Path.cwd() if (Path.cwd() / 'models').is_dir() else Path.cwd().parent
models_dir = repo_root / 'models'
if str(models_dir) not in sys.path:
    sys.path.insert(0, str(models_dir))
model_path = models_dir / '03_dummy_weekly_next_week.py'
spec = spec_from_file_location('dummy_weekly_model', model_path)
dummy_model = module_from_spec(spec)
assert spec and spec.loader
sys.modules[spec.name] = dummy_model
spec.loader.exec_module(dummy_model)

## 공통 지표 해석

- **Recall@Top-20%**: 위험확률 상위 20%가 전체 실제 이탈자의 몇 %를 포착하는지 봅니다. Dummy의 Fold 내부 확률은 모두 같으므로 이 값은 순위 성능이 아니라 **동률 처리와 안정 정렬된 행 순서의 영향을 받는 참고값**입니다.
- **PR-AUC**: 양성이 희소한 문제에서 무작위 순위 기준은 대략 양성률과 연결됩니다. Dummy가 이 관계를 확인하는 기준점입니다.
- **Brier Score**: 확률과 0/1 정답의 제곱오차 평균이며 낮을수록 좋습니다.
- **ECE 10-bin**: 확률 구간별 평균 예측과 실제 양성률의 차이를 표본 비율로 가중합니다. 낮을수록 보정이 좋습니다.

In [ ]:
# 전체 학습을 다시 실행할 때만 True로 바꿉니다. 기존 CSV 조회는 즉시 실행됩니다.
RUN_TRAINING = False
output_dir = repo_root / 'models' / 'demo_1'
if RUN_TRAINING:
    data_path = dummy_model.resolve_data_path(None)
    result = dummy_model.run_dummy(data_path, output_dir=output_dir)
    metrics = result['metrics']
    fold_metrics = result['fold_metrics']
else:
    metrics = pd.read_csv(output_dir / 'dummy_weekly_next_week_metrics.csv')
    fold_metrics = pd.read_csv(output_dir / 'dummy_weekly_next_week_fold_metrics.csv')
display(metrics)
display(fold_metrics)

## 해석 시 주의

각 Fold는 학습 Fold의 prior만 사용하므로 검증 Fold의 이탈률을 미리 보지 않습니다. Fold별 prior가 미세하게 달라 전체 OOF에는 최대 세 개 확률이 존재할 수 있지만, 한 Fold 안에서는 모든 확률이 같습니다. 따라서 Dummy의 Recall@Top-20%를 실질적인 개인별 위험 순위로 해석하면 안 됩니다.